# VisoLabel Colab Trainer

Use this notebook when VisoLabel gives you a Colab training token. The notebook opens a GUI where you can paste the token, inspect the resolved training bundle, start training, watch progress, and upload the checkpoints back to VisoLabel.

## How to use it

1. In VisoLabel, choose `Train on Colab` and copy the generated token.
2. In Colab, set the runtime to GPU from `Runtime > Change runtime type`.
3. Run the single cell below named `Launch VisoLabel Training GUI`.
4. Paste the token into the GUI and click `Run full pipeline`.
5. Keep the tab open until the status changes to `Complete`.

The token is a secret. For encrypted handoffs it contains the server token and the decryption password, so do not share it.

## GUI features

- Token input field with optional API URL override for private API deployments.
- Bundle preview showing task, model, epochs, batch size, image size, class count, and dataset image count.
- One-click full pipeline plus separate resolve, train, and upload controls for troubleshooting.
- Progress bar, status cards, and live logs for download, decryption, extraction, training, and upload.


In [ ]:
#@title Launch VisoLabel Training GUI { display-mode: "form" }
#@markdown Run this cell once. The implementation is hidden so the notebook stays focused on the GUI.
!pip -q install requests tqdm cryptography ipywidgets

import base64
import hashlib
import html
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import requests
from IPython.display import HTML, clear_output, display
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from tqdm.auto import tqdm

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

import ipywidgets as widgets

DEFAULT_API_BASE_URL = os.environ.get("VISOLABEL_API_BASE", "https://api.visolabel.ovh")
TOKEN_PREFIX = "VL1."
LEGACY_TOKEN_PREFIX = "VL-COLAB-ENC1."
WORK_DIR = Path("/content/visolabel_run")
BUNDLE_ENCRYPTED = WORK_DIR / "bundle.zip.vlenc"
BUNDLE_ZIP = WORK_DIR / "bundle.zip"
BUNDLE_DIR = WORK_DIR / "bundle"
OUTPUT_DIR = WORK_DIR / "output"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
ARTIFACT_EXTENSIONS = {".pth", ".pt", ".onnx", ".log", ".json", ".yaml", ".yml", ".txt"}

for directory in (WORK_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def _b64url_decode(value: str) -> bytes:
    return base64.urlsafe_b64decode((value + "=" * (-len(value) % 4)).encode("ascii"))


def _b64decode(value: str) -> bytes:
    return base64.b64decode(value.encode("ascii"), validate=True)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _normalize_encryption_metadata(value: dict) -> dict:
    if not isinstance(value, dict) or not value:
        return {}
    if value.get("salt_b64") and value.get("nonce_b64") and value.get("tag_b64"):
        return value
    metadata = {
        "enabled": True,
        "algorithm": value.get("algorithm", "AES-256-GCM"),
        "kdf": value.get("kdf", "PBKDF2-HMAC-SHA256"),
        "kdf_iterations": value.get("kdf_iterations") or value.get("i") or 200000,
        "salt_b64": value.get("s") or value.get("salt") or value.get("salt_b64", ""),
        "nonce_b64": value.get("n") or value.get("nonce") or value.get("nonce_b64", ""),
        "tag_b64": value.get("g") or value.get("tag") or value.get("tag_b64", ""),
        "aad_b64": value.get("a") or value.get("aad_b64") or base64.b64encode(b"visolabel-colab-bundle-v1").decode("ascii"),
        "plaintext_sha256": value.get("ps") or value.get("plaintext_sha256", ""),
        "ciphertext_sha256": value.get("cs") or value.get("ciphertext_sha256", ""),
        "plaintext_size": value.get("psz") or value.get("plaintext_size", 0),
        "ciphertext_size": value.get("csz") or value.get("ciphertext_size", 0),
    }
    if not metadata["salt_b64"] or not metadata["nonce_b64"] or not metadata["tag_b64"]:
        return {}
    return metadata


def _extract_bundle_url(value: dict) -> str:
    if not isinstance(value, dict):
        return ""
    for key in ("bundle_url", "download_url", "presigned_download_url", "s3_url", "url"):
        found = str(value.get(key) or "")
        if found:
            return found
    for key in ("bundle", "upload", "data", "result"):
        found = _extract_bundle_url(value.get(key) or {})
        if found:
            return found
    return ""


def _extract_encryption_metadata(value: dict) -> dict:
    if not isinstance(value, dict):
        return {}
    direct = _normalize_encryption_metadata(value.get("encryption") or value.get("encryption_metadata") or value.get("e") or {})
    if direct:
        return direct
    metadata = _normalize_encryption_metadata(value)
    if metadata:
        return metadata
    for key in ("bundle", "upload", "data", "result"):
        metadata = _extract_encryption_metadata(value.get(key) or {})
        if metadata:
            return metadata
    return {}


def _safe_extract_zip(zip_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if destination != target and destination not in target.parents:
                raise RuntimeError(f"Unsafe path in bundle: {member.filename}")
        zf.extractall(destination)


def _install_package(package: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


def _count_images(root: Path) -> int:
    if not root.exists():
        return 0
    return sum(1 for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def _format_bytes(value: int) -> str:
    size = float(value)
    for unit in ("B", "KB", "MB", "GB"):
        if size < 1024 or unit == "GB":
            return f"{size:.1f} {unit}" if unit != "B" else f"{int(size)} B"
        size /= 1024


class VisoLabelColabGUI:
    def __init__(self):
        self.api_base_url = DEFAULT_API_BASE_URL
        self.token_value = ""
        self.api_token = ""
        self.manifest = None
        self.cfg = {}
        self.task = ""
        self.command = ""
        self.dataset_dir = None
        self.upload_list = []
        self.encrypted = False
        self.training_finished = False
        self.started_at = None

        self.token = widgets.Password(
            description="Token",
            placeholder="Paste the token copied from VisoLabel",
            layout=widgets.Layout(width="100%"),
            style={"description_width": "70px"},
        )
        self.api_url = widgets.Text(
            value=DEFAULT_API_BASE_URL,
            description="API URL",
            layout=widgets.Layout(width="100%"),
            style={"description_width": "70px"},
        )
        self.progress = widgets.FloatProgress(
            value=0,
            min=0,
            max=100,
            description="Progress",
            bar_style="",
            layout=widgets.Layout(width="100%"),
            style={"description_width": "75px"},
        )
        self.status = widgets.HTML()
        self.summary = widgets.HTML()
        self.log_output = widgets.Output(layout=widgets.Layout(border="1px solid #d0d7de", height="320px", overflow_y="auto"))

        self.run_button = widgets.Button(description="Run full pipeline", button_style="primary", layout=widgets.Layout(width="170px"))
        self.resolve_button = widgets.Button(description="Resolve bundle", layout=widgets.Layout(width="145px"))
        self.train_button = widgets.Button(description="Start training", layout=widgets.Layout(width="145px"), disabled=True)
        self.upload_button = widgets.Button(description="Upload results", layout=widgets.Layout(width="145px"), disabled=True)
        self.clear_button = widgets.Button(description="Clear logs", layout=widgets.Layout(width="115px"))

        self.run_button.on_click(self._on_run_full)
        self.resolve_button.on_click(self._on_resolve)
        self.train_button.on_click(self._on_train)
        self.upload_button.on_click(self._on_upload)
        self.clear_button.on_click(lambda _: self.log_output.clear_output())

        self._set_status("Waiting for token", "Paste a VisoLabel token, then run the full pipeline.", "idle")
        self._render_summary()

    def display(self):
        clear_output(wait=True)
        display(HTML(self._style()))
        display(widgets.VBox([
            widgets.HTML(self._header()),
            widgets.VBox([self.token, self.api_url], layout=widgets.Layout(gap="8px")),
            widgets.HBox([self.run_button, self.resolve_button, self.train_button, self.upload_button, self.clear_button], layout=widgets.Layout(gap="8px", flex_flow="row wrap")),
            self.status,
            self.progress,
            self.summary,
            widgets.HTML("<div class='vl-section-title'>Live logs</div>"),
            self.log_output,
        ], layout=widgets.Layout(gap="12px")))

    def _style(self):
        return """
        <style>
        .vl-hero {background:#111827;color:#ffffff;padding:18px 20px;border-radius:8px;border-left:6px solid #14b8a6;box-shadow:0 8px 24px rgba(17,24,39,.18);}
        .vl-hero h2 {font-size:24px;line-height:1.2;margin:0 0 6px 0;letter-spacing:0;}
        .vl-hero p {font-size:14px;line-height:1.45;margin:0;color:#d1d5db;}
        .vl-section-title {font-size:13px;font-weight:700;color:#374151;margin-top:6px;text-transform:uppercase;letter-spacing:.04em;}
        .vl-status {border:1px solid #d0d7de;border-radius:8px;padding:12px 14px;background:#ffffff;}
        .vl-status strong {font-size:15px;color:#111827;}
        .vl-status span {display:block;font-size:13px;color:#4b5563;margin-top:3px;}
        .vl-idle {border-left:5px solid #64748b;}
        .vl-active {border-left:5px solid #2563eb;}
        .vl-ok {border-left:5px solid #16a34a;}
        .vl-error {border-left:5px solid #dc2626;}
        .vl-grid {display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:10px;margin-top:2px;}
        .vl-card {border:1px solid #d0d7de;border-radius:8px;padding:12px;background:#ffffff;min-height:68px;}
        .vl-label {font-size:12px;color:#6b7280;text-transform:uppercase;letter-spacing:.04em;margin-bottom:5px;}
        .vl-value {font-size:16px;font-weight:700;color:#111827;word-break:break-word;}
        .vl-muted {font-size:12px;color:#6b7280;margin-top:4px;}
        </style>
        """

    def _header(self):
        return """
        <div class='vl-hero'>
          <h2>VisoLabel Colab Trainer</h2>
          <p>Paste a token, preview the bundle, train on the Colab GPU, and send checkpoints back to VisoLabel.</p>
        </div>
        """

    def _set_status(self, title: str, detail: str = "", kind: str = "active"):
        title = html.escape(title)
        detail = html.escape(detail)
        self.status.value = f"<div class='vl-status vl-{kind}'><strong>{title}</strong><span>{detail}</span></div>"

    def _render_summary(self):
        cfg = self.cfg or {}
        classes = cfg.get("classes") or []
        if not isinstance(classes, list):
            classes = []
        dataset_count = _count_images(self.dataset_dir) if self.dataset_dir else 0
        model_name = cfg.get("model_name", "-")
        epochs = cfg.get("epochs", "-")
        batch_size = cfg.get("batch_size", "-")
        image_size = cfg.get("image_size", "-")
        task_label = self.task or "Not resolved"
        security = "Encrypted" if self.encrypted else ("Legacy token" if self.manifest else "-")
        values = [
            ("Task", task_label, self.command or "Waiting"),
            ("Model", model_name, security),
            ("Epochs", epochs, f"Batch {batch_size}"),
            ("Image size", image_size, f"{len(classes)} classes"),
            ("Dataset", f"{dataset_count} images", str(self.dataset_dir or "-")),
        ]
        cards = []
        for label, value, muted in values:
            cards.append(
                "<div class='vl-card'>"
                f"<div class='vl-label'>{html.escape(str(label))}</div>"
                f"<div class='vl-value'>{html.escape(str(value))}</div>"
                f"<div class='vl-muted'>{html.escape(str(muted))}</div>"
                "</div>"
            )
        self.summary.value = "<div class='vl-grid'>" + "".join(cards) + "</div>"

    def _log(self, message: str):
        with self.log_output:
            print(f"[{time.strftime('%H:%M:%S')}] {message}")

    def _post(self, path: str, payload: dict) -> dict:
        url = f"{self.api_base_url.rstrip('/')}{path}"
        response = requests.post(url, json=payload, timeout=120)
        response.raise_for_status()
        return response.json()

    def _reset_run_files(self):
        for path in (BUNDLE_ENCRYPTED, BUNDLE_ZIP):
            try:
                if path.exists():
                    path.unlink()
            except Exception as exc:
                self._log(f"Could not remove {path.name}: {exc}")
        for path in (BUNDLE_DIR, OUTPUT_DIR):
            try:
                if path.exists():
                    shutil.rmtree(path)
                path.mkdir(parents=True, exist_ok=True)
            except Exception as exc:
                self._log(f"Could not reset {path}: {exc}")

    def _download(self, url: str, destination: Path, start: float, end: float):
        with requests.get(url, stream=True, timeout=600) as response:
            response.raise_for_status()
            total = int(response.headers.get("content-length") or 0)
            done = 0
            with open(destination, "wb") as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    f.write(chunk)
                    done += len(chunk)
                    if total:
                        self.progress.value = start + min(done / total, 1.0) * (end - start)
            if total:
                self._log(f"Downloaded {_format_bytes(done)}.")
            else:
                self._log(f"Downloaded {_format_bytes(destination.stat().st_size)}.")

    def _decrypt_bundle(self, encrypted_path: Path, output_path: Path, password: str, encryption: dict):
        if encryption.get("algorithm") != "AES-256-GCM":
            raise RuntimeError(f"Unsupported encryption algorithm: {encryption.get('algorithm')}")
        if encryption.get("kdf") != "PBKDF2-HMAC-SHA256":
            raise RuntimeError(f"Unsupported KDF: {encryption.get('kdf')}")

        salt = _b64decode(encryption["salt_b64"])
        nonce = _b64decode(encryption["nonce_b64"])
        tag = _b64decode(encryption["tag_b64"])
        aad = _b64decode(encryption.get("aad_b64", base64.b64encode(b"visolabel-colab-bundle-v1").decode("ascii")))
        iterations = int(encryption.get("kdf_iterations", 200000))
        kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=iterations)
        key = kdf.derive(password.encode("utf-8"))

        decryptor = Cipher(algorithms.AES(key), modes.GCM(nonce, tag)).decryptor()
        decryptor.authenticate_additional_data(aad)
        with open(encrypted_path, "rb") as src, open(output_path, "wb") as dst:
            for chunk in iter(lambda: src.read(1024 * 1024), b""):
                data = decryptor.update(chunk)
                if data:
                    dst.write(data)
            data = decryptor.finalize()
            if data:
                dst.write(data)

    def _extract_manifest(self):
        if BUNDLE_DIR.exists():
            shutil.rmtree(BUNDLE_DIR)
        BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
        _safe_extract_zip(BUNDLE_ZIP, BUNDLE_DIR)

        manifest_path = BUNDLE_DIR / "training_bundle.json"
        with open(manifest_path, "r", encoding="utf-8") as f:
            self.manifest = json.load(f)

        self.task = self.manifest["task"]["task_type"]
        self.command = self.manifest["task"]["command"]
        self.cfg = self.manifest["training_config"]
        self.dataset_dir = BUNDLE_DIR / self.manifest["paths"]["dataset_dir"]
        self.train_button.disabled = False
        self._render_summary()

    def resolve_bundle(self):
        self.token_value = self.token.value.strip() or os.environ.get("VISOLABEL_TOKEN", "").strip()
        if not self.token_value:
            raise RuntimeError("Paste a VisoLabel token first.")

        self.api_base_url = self.api_url.value.strip() or DEFAULT_API_BASE_URL
        self.api_token = self.token_value
        self.encrypted = False
        self.training_finished = False
        self.started_at = time.time()
        self.progress.value = 0
        self._reset_run_files()
        self._set_status("Resolving bundle", "Downloading bundle metadata and preparing the dataset.", "active")
        self._log("Resolving token.")

        if self.token_value.startswith(TOKEN_PREFIX) or self.token_value.startswith(LEGACY_TOKEN_PREFIX):
            prefix = TOKEN_PREFIX if self.token_value.startswith(TOKEN_PREFIX) else LEGACY_TOKEN_PREFIX
            payload = json.loads(_b64url_decode(self.token_value[len(prefix):]).decode("utf-8"))
            if isinstance(payload, list):
                bundle_id = ""
                self.api_token = str(payload[0]) if len(payload) >= 1 else ""
                password = str(payload[1]) if len(payload) >= 2 else ""
                bundle_url = ""
                encryption = {}
                if len(payload) >= 3:
                    if isinstance(payload[2], str):
                        self.api_base_url = payload[2].rstrip("/") if payload[2] else self.api_base_url
                    elif isinstance(payload[2], dict):
                        encryption = _normalize_encryption_metadata(payload[2])
                if len(payload) >= 4 and isinstance(payload[3], dict):
                    encryption = _normalize_encryption_metadata(payload[3])
                self.api_url.value = self.api_base_url
            elif isinstance(payload, dict):
                self.api_base_url = payload.get("api_base_url") or payload.get("u") or self.api_base_url
                self.api_url.value = self.api_base_url
                has_explicit_server_token = "server_token" in payload or "t" in payload
                bundle_id = payload.get("bundle_id") or payload.get("i") or (payload.get("b") if has_explicit_server_token else "") or ""
                self.api_token = payload.get("server_token") or payload.get("t") or payload.get("b") or ""
                bundle_url = payload.get("bundle_url") or ""
                password = payload.get("password") or payload.get("p") or ""
                encryption = _extract_encryption_metadata(payload)
            else:
                raise RuntimeError("Unsupported encrypted VisoLabel token payload.")
            self.encrypted = True
            if not self.api_token:
                raise RuntimeError("server_token or bundle_id missing in encrypted VisoLabel token.")
            if not bundle_url or not encryption:
                resolve_payload = {"token": self.api_token}
                if bundle_id:
                    resolve_payload["bundle_id"] = bundle_id
                resolve = self._post("/api/v1/colab/resolve-token/", resolve_payload)
                bundle_url = bundle_url or _extract_bundle_url(resolve)
                encryption = encryption or _extract_encryption_metadata(resolve)
            if not bundle_url or not password or not encryption:
                missing = []
                if not bundle_url:
                    missing.append("bundle_url")
                if not password:
                    missing.append("password")
                if not encryption:
                    missing.append("encryption metadata")
                raise RuntimeError("Encrypted token could not resolve " + ", ".join(missing) + ". Regenerate the token with the latest VisoLabel build or update the resolve-token API to return encryption metadata.")

            self.progress.value = 5
            self._log("Downloading encrypted bundle.")
            self._download(bundle_url, BUNDLE_ENCRYPTED, 5, 30)
            expected_cipher_sha = encryption.get("ciphertext_sha256", "")
            if expected_cipher_sha and _sha256(BUNDLE_ENCRYPTED) != expected_cipher_sha:
                raise RuntimeError("Encrypted bundle SHA-256 does not match token metadata.")

            self._set_status("Decrypting bundle", "The dataset is decrypted locally in this Colab runtime.", "active")
            self._log("Decrypting bundle locally.")
            self.progress.value = 35
            self._decrypt_bundle(BUNDLE_ENCRYPTED, BUNDLE_ZIP, password, encryption)
            expected_plain_sha = encryption.get("plaintext_sha256", "")
            if expected_plain_sha and _sha256(BUNDLE_ZIP) != expected_plain_sha:
                raise RuntimeError("Decrypted bundle SHA-256 does not match token metadata.")
        else:
            resolve = self._post("/api/v1/colab/resolve-token/", {"token": self.token_value})
            bundle_url = resolve.get("bundle_url", "")
            if not bundle_url:
                raise RuntimeError("bundle_url missing in resolve-token response.")
            self.progress.value = 10
            self._log("Downloading bundle.")
            self._download(bundle_url, BUNDLE_ZIP, 10, 40)

        self._set_status("Extracting bundle", "Reading the manifest and dataset layout.", "active")
        self.progress.value = 45
        self._extract_manifest()
        self.progress.value = 50
        self._log(f"Resolved {self.task} bundle with command '{self.command}'.")
        self._set_status("Bundle ready", "Review the preview, then start training.", "ok")

    def train(self):
        if not self.manifest:
            self.resolve_bundle()
        self.upload_button.disabled = True
        self._set_status("Training", "Training output appears in the logs below.", "active")
        self.progress.value = max(self.progress.value, 52)

        if self.command == "train":
            self._train_rfdetr()
        elif self.command == "classify":
            self._train_classifier()
        else:
            raise RuntimeError(f"Unsupported command: {self.command}")

        self.progress.value = 90
        self.training_finished = True
        self.upload_button.disabled = False
        self._set_status("Training finished", f"Outputs are in {OUTPUT_DIR}.", "ok")
        self._log(f"Training finished. Output dir: {OUTPUT_DIR}")

    def _train_rfdetr(self):
        self._log("Installing RF-DETR dependencies.")
        _install_package("rfdetr[train,loggers]")
        os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
        import torch
        from rfdetr import RFDETRBase, RFDETRLarge
        from rfdetr import RFDETRSegLarge, RFDETRSegMedium, RFDETRSegNano, RFDETRSegSmall

        model_name = str(self.cfg["model_name"])
        task_type = self.manifest["task"]["task_type"]
        is_segmentation = task_type == "instance_segmentation"
        resolution = int(self.cfg.get("image_size", 560))
        requested_batch_size = int(self.cfg.get("batch_size", 4))
        batch_size = min(requested_batch_size, 2) if is_segmentation else requested_batch_size
        if batch_size != requested_batch_size:
            self._log(f"Using Colab-safe segmentation batch size {batch_size} instead of requested {requested_batch_size}.")
        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        self._log(f"Runtime device: {gpu_name}; resolution={resolution}; batch_size={batch_size}; AMP disabled.")

        if is_segmentation:
            if model_name.endswith("-L"):
                model = RFDETRSegLarge(resolution=resolution, amp=False)
            elif model_name.endswith("-M"):
                model = RFDETRSegMedium(resolution=resolution, amp=False)
            elif model_name.endswith("-S"):
                model = RFDETRSegSmall(resolution=resolution, amp=False)
            else:
                model = RFDETRSegNano(resolution=resolution, amp=False)
        else:
            model = RFDETRLarge(resolution=resolution, amp=False) if model_name.endswith("-L") else RFDETRBase(resolution=resolution, amp=False)

        self._log(f"Starting {model_name} training. RF-DETR will show detailed progress below.")
        # IMPORTANT: do NOT wrap model.train() in `with self.log_output:`.
        # Lightning's progress bar (tqdm/Rich Live) uses carriage-return
        # updates and ANSI cursor controls that the ipywidgets Output
        # widget cannot render — output buffers fill up and the run
        # appears frozen after "Loading train_dataloader to estimate
        # number of stepping batches". We also disable tqdm entirely
        # (progress_bar=False) for the same reason the local backend
        # does it: CR-only writes deadlock on captured streams.
        train_kwargs = dict(
            dataset_dir=str(self.dataset_dir),
            epochs=int(self.cfg.get("epochs", 50)),
            batch_size=batch_size,
            lr=float(self.cfg.get("learning_rate", 1e-4)),
            output_dir=str(OUTPUT_DIR),
            class_names=self.cfg.get("classes", []),
            num_workers=0,
            persistent_workers=False,
            multi_scale=False,
            expanded_scales=False,
            compute_val_loss=False,
            compute_test_loss=False,
            log_every_n_steps=1,
            tensorboard=False,
            progress_bar=False,
        )
        total_epochs = int(train_kwargs["epochs"])
        try:
            model.callbacks["on_fit_epoch_end"].append(
                lambda info: (
                    self._log(
                        f"Epoch {int(info.get('epoch', 0)) + 1}/{total_epochs} - "
                        + ', '.join(
                            f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
                            for k, v in info.items() if k != 'epoch'
                        )
                    ),
                    setattr(self.progress, 'value', 55 + ((int(info.get('epoch', 0)) + 1) / max(total_epochs, 1)) * 33),
                )
            )
        except Exception as cb_exc:
            self._log(f"Could not register epoch callback: {cb_exc}")
        try:
            model.train(**train_kwargs)
        except TypeError as exc:
            # Older rfdetr versions may not accept progress_bar=False; retry without it.
            if "progress_bar" in str(exc):
                self._log("rfdetr does not support progress_bar=False; retrying without it.")
                train_kwargs.pop("progress_bar", None)
                model.train(**train_kwargs)
            else:
                raise

    def _train_classifier(self):
        self._log("Installing classifier dependencies.")
        _install_package("timm")
        import timm
        import torch
        import torch.nn as nn
        import torch.optim as optim
        from torch.utils.data import DataLoader
        from torchvision import datasets, transforms

        image_size = int(self.cfg.get("image_size", 224))
        batch_size = int(self.cfg.get("batch_size", 16))
        epochs = int(self.cfg.get("epochs", 30))
        train_ds = datasets.ImageFolder(str(self.dataset_dir / "train"), transform=transforms.Compose([
            transforms.RandomResizedCrop(image_size),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
        ]))
        valid_path = self.dataset_dir / "valid"
        val_ds = datasets.ImageFolder(str(valid_path), transform=transforms.Compose([
            transforms.Resize(int(image_size * 1.14)),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
        ])) if valid_path.exists() else None

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2) if val_ds else None
        model_name = str(self.cfg.get("model_name", "convnext_tiny.fb_in22k_ft_in1k"))
        model = timm.create_model(model_name, pretrained=True, num_classes=len(train_ds.classes))
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = model.to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(model.parameters(), lr=float(self.cfg.get("learning_rate", 1e-4)))

        best_path = OUTPUT_DIR / "checkpoint_best_total.pth"
        best_acc = 0.0
        self._log(f"Starting {model_name} classification training on {device}.")
        for epoch in range(epochs):
            model.train()
            epoch_loss = 0.0
            with self.log_output:
                batches = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}", leave=False)
                for x, y in batches:
                    x, y = x.to(device), y.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(x), y)
                    loss.backward()
                    optimizer.step()
                    epoch_loss += float(loss.item())
                    batches.set_postfix(loss=f"{loss.item():.4f}")

            if val_loader:
                model.eval()
                total = 0
                correct = 0
                with torch.no_grad():
                    for x, y in val_loader:
                        x, y = x.to(device), y.to(device)
                        pred = model(x).argmax(1)
                        total += y.shape[0]
                        correct += (pred == y).sum().item()
                acc = (correct / max(total, 1)) * 100.0
                if acc >= best_acc:
                    best_acc = acc
                    torch.save({"model_state_dict": model.state_dict(), "classes": train_ds.classes}, best_path)
                self._log(f"Epoch {epoch + 1}/{epochs}: validation accuracy {acc:.2f}%")
            else:
                self._log(f"Epoch {epoch + 1}/{epochs}: loss {epoch_loss / max(len(train_loader), 1):.4f}")

            self.progress.value = 55 + ((epoch + 1) / max(epochs, 1)) * 35

        if not best_path.exists():
            torch.save({"model_state_dict": model.state_dict(), "classes": train_ds.classes}, best_path)

    def _request_upload(self, path: Path) -> dict:
        return self._post("/api/v1/colab/request-checkpoint-upload/", {
            "token": self.api_token,
            "file_name": path.name,
            "file_size": path.stat().st_size,
        })

    def _upload_artifact(self, path: Path, index: int, total: int):
        request = self._request_upload(path)
        upload_url = request.get("upload_url", "")
        if not upload_url:
            raise RuntimeError(f"upload_url missing for {path.name}")
        method = str(request.get("upload_method", "PUT")).upper()
        headers = request.get("upload_headers", {}) or {}
        headers.pop("Host", None)
        self._log(f"Uploading {path.name} ({_format_bytes(path.stat().st_size)}).")
        with open(path, "rb") as f:
            if method == "POST":
                response = requests.post(upload_url, files={"file": (path.name, f)}, headers=headers, timeout=600)
            else:
                response = requests.put(upload_url, data=f, headers=headers, timeout=600)
        response.raise_for_status()
        self.progress.value = 90 + (index / max(total, 1)) * 8

    def upload(self):
        if not self.api_token:
            raise RuntimeError("Resolve the bundle before uploading results.")
        self._set_status("Uploading results", "Sending checkpoints and logs back to VisoLabel.", "active")
        self.upload_list = sorted(p for p in OUTPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in ARTIFACT_EXTENSIONS)
        if not self.upload_list:
            self._log("No uploadable artifacts found.")
        for index, path in enumerate(self.upload_list, start=1):
            self._upload_artifact(path, index, len(self.upload_list))
        try:
            self._post("/api/v1/colab/finalize-run/", {"token": self.api_token})
        except Exception as exc:
            self._log(f"Finalize endpoint not available: {exc}")
        elapsed = int(time.time() - self.started_at) if self.started_at else 0
        self.progress.value = 100
        self._set_status("Complete", f"Pipeline finished in {elapsed // 60}m {elapsed % 60}s.", "ok")
        self._log("Upload complete.")

    def _set_busy(self, busy: bool):
        self.run_button.disabled = busy
        self.resolve_button.disabled = busy
        self.train_button.disabled = busy or self.manifest is None
        self.upload_button.disabled = busy or not self.api_token or not self.training_finished

    def _run_action(self, action):
        self._set_busy(True)
        try:
            action()
        except Exception as exc:
            self._set_status("Error", str(exc), "error")
            self._log(f"Error: {exc}")
            raise
        finally:
            self._set_busy(False)
            self._render_summary()

    def _on_resolve(self, _):
        self._run_action(self.resolve_bundle)

    def _on_train(self, _):
        self._run_action(self.train)

    def _on_upload(self, _):
        self._run_action(self.upload)

    def _on_run_full(self, _):
        def pipeline():
            self.resolve_bundle()
            self.train()
            self.upload()
        self._run_action(pipeline)


gui = VisoLabelColabGUI()
gui.display()
